# GPT4All 로컬 모델


> 업데이트 기준: **2026-09-18**  
> 책의 학습 목표는 유지하면서 LangChain 1.x의 분리된 provider 패키지와 현재 메시지/스트리밍 API에 맞췄습니다. 모델 이름은 공급자 정책에 따라 바뀔 수 있으므로 환경 변수로 덮어쓸 수 있게 구성했습니다.


GPT4All은 GGUF 모델을 로컬에서 실행합니다. LangChain 통합은 현재 `langchain-community`에 있습니다. 모델 파일은 GPT4All 앱의 Model Explorer 또는 신뢰할 수 있는 모델 배포처에서 내려받고 라이선스·RAM 요구량을 확인하세요.


In [ ]:
%pip install -qU langchain-community langchain-core gpt4all python-dotenv


In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
model_path = Path(
    os.getenv(
        "GPT4ALL_MODEL_PATH",
        "models/EEVE-Korean-Instruct-10.8B-v1.0-Q8_0.gguf",
    )
)

if not model_path.is_file():
    raise FileNotFoundError(
        "GGUF 모델 경로를 GPT4ALL_MODEL_PATH에 설정하세요: "
        f"{model_path.resolve()}"
    )


## 모델 로드와 스트리밍

`backend="gpu"`처럼 플랫폼 의존 값을 고정하지 않습니다. 먼저 CPU 기본값으로 확인한 뒤, 설치된 GPT4All 런타임 문서에 맞춰 장치 옵션을 추가하세요.


In [ ]:
from langchain_community.llms import GPT4All

llm = GPT4All(
    model=str(model_path),
    n_threads=max(1, (os.cpu_count() or 2) - 1),
    max_tokens=512,
    temp=0,
    streaming=True,
    verbose=False,
)

for chunk in llm.stream("대한민국의 수도는 어디인가요?"):
    print(chunk, end="", flush=True)


## LCEL 체인

GPT4All은 문자열 LLM이므로 `ChatPromptTemplate`보다 일반 `PromptTemplate`이 모델별 템플릿을 명시하기 쉽습니다. 실제 프롬프트 형식은 선택한 GGUF 모델 카드에 맞춰 조정합니다.


In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    "당신은 친절한 한국어 도우미입니다.\n\n"
    "질문: {question}\n"
    "답변:"
)
chain = prompt | llm | StrOutputParser()

print(chain.invoke({"question": "RAG를 한 문장으로 설명해 주세요."}))
